# Defining Reaction Mechanisms in Soft Potato 3.0: A Step-by-Step Tutorial

Welcome to this tutorial on defining electrochemical and chemical reaction mechanisms in **Soft Potato 3.0**.

Reaction mechanisms are the foundation of electrochemical modeling and simulation. In this notebook, you will learn how to:
- Define chemical and electroactive species with strict physical units.
- Formulate interfacial electrochemical reactions (heterogeneous electron transfers).
- Formulate bulk chemical reactions (homogeneous equilibria and kinetics).
- Combine reactions into a comprehensive `Mechanism`.
- Implement classic electrochemical mechanisms: **E**, **EC**, **CE**, **EE**, **ECE**, and **DISP**.
- Inspect key properties such as homogeneous stoichiometry matrices and concentration profiles.

## Mechanism Definitions Overview

The table below summarizes the classic mechanisms implemented in this tutorial:

| Mechanism | Reaction Scheme | Description |
| :--- | :--- | :--- |
| **E** | $\text{O} + \text{e}^- \rightleftharpoons \text{R}$ | Simple single electron transfer at the electrode. |
| **EC** | $\text{O} + \text{e}^- \rightleftharpoons \text{R}$<br>$\text{R} \rightleftharpoons \text{Z}$ | Electron transfer followed by a homogeneous chemical reaction consuming product $\text{R}$. |
| **CE** | $\text{Y} \rightleftharpoons \text{O}$<br>$\text{O} + \text{e}^- \rightleftharpoons \text{R}$ | Preceding homogeneous equilibrium producing the electroactive species $\text{O}$ from precursor $\text{Y}$. |
| **EE** | $\text{O} + \text{e}^- \rightleftharpoons \text{R}$<br>$\text{R} + \text{e}^- \rightleftharpoons \text{P}$ | Two sequential electron transfers with distinct standard potentials ($E_{0,1}$ and $E_{0,2}$). |
| **ECE** | $\text{O} + \text{e}^- \rightleftharpoons \text{R}$<br>$\text{R} \rightleftharpoons \text{Y}$<br>$\text{Y} + \text{e}^- \rightleftharpoons \text{Z}$ | Heterogeneous electron transfer, chemical transformation of intermediate, second heterogeneous electron transfer. |
| **DISP (ECE Competition)** | $\text{O} + \text{e}^- \rightleftharpoons \text{R}$<br>$\text{R} \rightleftharpoons \text{Y}$<br>$\text{R} + \text{Y} \rightleftharpoons \text{O} + \text{Z}$ | Intermediate $\text{Y}$ is reduced homogeneously in the diffusion layer by species $\text{R}$, regenerating $\text{O}$. |
| **DISP (Radical Disproportionation)** | $\text{O} + \text{e}^- \rightleftharpoons \text{R}$<br>$2\text{R} \rightleftharpoons \text{O} + \text{Z}$ | Two molecules of intermediate $\text{R}$ disproportionate in solution to regenerate $\text{O}$ and form product $\text{Z}$. |

> **Note on Units (CGS System)**:
> Soft Potato strictly enforces CGS units across all modules:
> - **Diffusion coefficient ($D$)**: $\text{cm}^2/\text{s}$ (typically $\sim 10^{-5}\text{ cm}^2/\text{s}$ for small molecules in water).
> - **Bulk concentration ($c_{\text{bulk}}$)**: $\text{mol}/\text{cm}^3$ ($1\text{ mM} = 10^{-3}\text{ mol/L} = 10^{-6}\text{ mol/cm}^3$).
> - **Standard potential ($E_0$)**: $\text{V}$ vs. reference electrode.

### Installation & Environment Setup

If you are running this notebook in **Google Colab**, **Kaggle**, or a new environment where Soft Potato and its dependencies (`numpy`, `scipy`, `matplotlib`) are not yet installed, run the cell below.

Soft Potato will be installed directly from the main GitHub repository:
`https://github.com/oliverrdz/softpotato.git`

In [1]:
# Check and install Soft Potato and dependencies from the main repository if not already installed
try:
    import softpotato
except ImportError:
    print("Soft Potato not found. Installing from the main GitHub repository...")
    %pip install -q "git+https://github.com/oliverrdz/softpotato.git"
    import softpotato

print(f"Soft Potato version {softpotato.__version__} is ready to use!")

Soft Potato not found. Installing from the main GitHub repository...
Note: you may need to restart the kernel to use updated packages.
Soft Potato version 3.0.0.dev1 is ready to use!


In [2]:
import numpy as np
from softpotato.core import (
    Species,
    ElectrochemicalReaction,
    ChemicalReaction,
    Mechanism,
)

print("Soft Potato core components imported successfully!")

Soft Potato core components imported successfully!


## Step 1: Defining Chemical Species (`Species`)

The `Species` class represents an electroactive or chemical component in solution.

When creating a species, you specify:
- `name` (str): Identifier (e.g., `'O'`, `'R'`, `'Z'`).
- `D` (float): Diffusion coefficient in $\text{cm}^2/\text{s}$ (must be non-negative).
- `c_bulk` (float): Initial uniform bulk concentration in $\text{mol}/\text{cm}^3$ (default: `0.0`). Note that $1\text{ mM} = 10^{-6}\text{ mol}/\text{cm}^3$.
- `charge` (int): Optional ionic valence / charge $z$ (default: `0`).

In [3]:
# Define oxidized (O) and reduced (R) species
# 1 mM bulk concentration = 1.0e-6 mol/cm³
spec_O = Species(name="O", D=1.0e-5, c_bulk=1.0e-6, charge=0)
spec_R = Species(name="R", D=1.0e-5, c_bulk=0.0, charge=-1)

print(f"Species created: {spec_O}")
print(f"Representation: {repr(spec_O)}")
print(f"Bulk concentration: {spec_O.c_bulk} mol/cm³ ({spec_O.c_bulk * 1e6:.1f} mM)")

Species created: Species O (D=1.00e-05 cm²/s, c_bulk=1.00e-06 mol/cm³, z=0)
Representation: Species(name='O', D=1.00e-05 cm²/s, c_bulk=1.00e-06 mol/cm³, charge=0)
Bulk concentration: 1e-06 mol/cm³ (1.0 mM)


## Step 2: The E Mechanism (Single Electron Transfer)

$$\text{O} + \text{e}^- \rightleftharpoons \text{R} \quad (E_0 = 0.00\text{ V})$$

An electrochemical reaction is defined using `ElectrochemicalReaction`:
- `reactants`: Reactant species in the reduction direction (e.g. `[spec_O]`).
- `products`: Product species in the reduction direction (e.g. `[spec_R]`).
- `n_electrons`: Number of electrons transferred ($n \ge 1$, default: 1).
- `E0`: Standard reduction potential in Volts (default: 0.0 V).

We then pass the reaction into a `Mechanism` container.

In [4]:
# 1. Define species
spec_O = Species(name="O", D=1.0e-5, c_bulk=1.0e-6)
spec_R = Species(name="R", D=1.0e-5, c_bulk=0.0)

# 2. Define electrochemical reaction
rxn_E = ElectrochemicalReaction(
    reactants=[spec_O],
    products=[spec_R],
    n_electrons=1,
    E0=0.0,
)

# 3. Create mechanism
mech_E = Mechanism([rxn_E])

print(mech_E)
print("\nSpecies discovered:", mech_E.species_names)
print("Electrochemical reactions:", len(mech_E.e_reactions))
print("Chemical reactions:", len(mech_E.c_reactions))

Mechanism:
  Species: Species O (D=1.00e-05 cm²/s, c_bulk=1.00e-06 mol/cm³, z=0), Species R (D=1.00e-05 cm²/s, c_bulk=0.00e+00 mol/cm³, z=0)
  Electrochemical Reactions:
  O + e⁻ ⇌ R (E0 = 0.000 V)
  Chemical Reactions:
  None

Species discovered: ['O', 'R']
Electrochemical reactions: 1
Chemical reactions: 0


## Step 3: The EC Mechanism (Following Chemical Reaction)

$$\text{O} + \text{e}^- \rightleftharpoons \text{R} \quad (E_0 = 0.00\text{ V})$$
$$\text{R} \rightleftharpoons \text{Z}$$

In an EC mechanism, the product of the electron transfer ($\text{R}$) undergoes a homogeneous chemical reaction to yield species $\text{Z}$.

We define the bulk step using `ChemicalReaction`:
- `reactants`: Reactants in the forward chemical direction.
- `products`: Products in the forward chemical direction.

`Mechanism` automatically discovers all unique species across both reactions and computes the homogeneous stoichiometry matrix.

In [5]:
# 1. Define species
spec_O = Species(name="O", D=1.0e-5, c_bulk=1.0e-6)
spec_R = Species(name="R", D=1.0e-5, c_bulk=0.0)
spec_Z = Species(name="Z", D=1.0e-5, c_bulk=0.0)

# 2. Define reactions
rxn_E = ElectrochemicalReaction(reactants=[spec_O], products=[spec_R], n_electrons=1, E0=0.0)
rxn_C = ChemicalReaction(reactants=[spec_R], products=[spec_Z])

# 3. Create mechanism
mech_EC = Mechanism([rxn_E, rxn_C])

print(mech_EC)
print("\nHomogeneous Stoichiometry Matrix (ν_i,j):")
print("Rows (Species):", mech_EC.species_names)
print(mech_EC.homogeneous_stoichiometry_matrix)

Mechanism:
  Species: Species O (D=1.00e-05 cm²/s, c_bulk=1.00e-06 mol/cm³, z=0), Species R (D=1.00e-05 cm²/s, c_bulk=0.00e+00 mol/cm³, z=0), Species Z (D=1.00e-05 cm²/s, c_bulk=0.00e+00 mol/cm³, z=0)
  Electrochemical Reactions:
  O + e⁻ ⇌ R (E0 = 0.000 V)
  Chemical Reactions:
  R ⇌ Z

Homogeneous Stoichiometry Matrix (ν_i,j):
Rows (Species): ['O', 'R', 'Z']
[[ 0.]
 [-1.]
 [ 1.]]


## Step 4: The CE Mechanism (Preceding Chemical Reaction)

$$\text{Y} \rightleftharpoons \text{O}$$
$$\text{O} + \text{e}^- \rightleftharpoons \text{R} \quad (E_0 = 0.00\text{ V})$$

In a CE mechanism, the electroactive species $\text{O}$ is initially generated from an electro-inactive precursor $\text{Y}$ through a bulk chemical reaction.

In the bulk solution, the precursor $\text{Y}$ is present at initial concentration ($c_{\text{bulk}} = 1\text{ mM}$).

In [6]:
spec_Y = Species(name="Y", D=1.0e-5, c_bulk=1.0e-6)  # Precursor in bulk
spec_O = Species(name="O", D=1.0e-5, c_bulk=0.0)     # Electroactive species
spec_R = Species(name="R", D=1.0e-5, c_bulk=0.0)     # Reduction product

rxn_C = ChemicalReaction(reactants=[spec_Y], products=[spec_O])
rxn_E = ElectrochemicalReaction(reactants=[spec_O], products=[spec_R], n_electrons=1, E0=0.0)

mech_CE = Mechanism([rxn_C, rxn_E])
print(mech_CE)

Mechanism:
  Species: Species Y (D=1.00e-05 cm²/s, c_bulk=1.00e-06 mol/cm³, z=0), Species O (D=1.00e-05 cm²/s, c_bulk=0.00e+00 mol/cm³, z=0), Species R (D=1.00e-05 cm²/s, c_bulk=0.00e+00 mol/cm³, z=0)
  Electrochemical Reactions:
  O + e⁻ ⇌ R (E0 = 0.000 V)
  Chemical Reactions:
  Y ⇌ O


## Step 5: The EE Mechanism (Two Sequential Electron Transfers)

$$\text{O} + \text{e}^- \rightleftharpoons \text{R} \quad (E_{0,1} = 0.00\text{ V})$$
$$\text{R} + \text{e}^- \rightleftharpoons \text{P} \quad (E_{0,2} = -0.30\text{ V})$$

When a molecule can be reduced in two successive electron transfers, species $\text{R}$ acts as the product of the first reaction and the reactant of the second.

In [7]:
spec_O = Species(name="O", D=1.0e-5, c_bulk=1.0e-6)
spec_R = Species(name="R", D=1.0e-5, c_bulk=0.0)
spec_P = Species(name="P", D=1.0e-5, c_bulk=0.0)

rxn_E1 = ElectrochemicalReaction(reactants=[spec_O], products=[spec_R], n_electrons=1, E0=0.0)
rxn_E2 = ElectrochemicalReaction(reactants=[spec_R], products=[spec_P], n_electrons=1, E0=-0.30)

mech_EE = Mechanism([rxn_E1, rxn_E2])
print(mech_EE)
print(f"\nNumber of electrochemical reactions: {len(mech_EE.e_reactions)}")

Mechanism:
  Species: Species O (D=1.00e-05 cm²/s, c_bulk=1.00e-06 mol/cm³, z=0), Species R (D=1.00e-05 cm²/s, c_bulk=0.00e+00 mol/cm³, z=0), Species P (D=1.00e-05 cm²/s, c_bulk=0.00e+00 mol/cm³, z=0)
  Electrochemical Reactions:
  O + e⁻ ⇌ R (E0 = 0.000 V)
  R + e⁻ ⇌ P (E0 = -0.300 V)
  Chemical Reactions:
  None

Number of electrochemical reactions: 2


## Step 6: The ECE Mechanism (Electrochemical - Chemical - Electrochemical)

$$\text{O} + \text{e}^- \rightleftharpoons \text{R} \quad (E_{0,1} = 0.00\text{ V})$$
$$\text{R} \rightleftharpoons \text{Y}$$
$$\text{Y} + \text{e}^- \rightleftharpoons \text{Z} \quad (E_{0,2} = -0.20\text{ V})$$

In an ECE mechanism:
1. Primary reduction generates $\text{R}$ at the electrode surface.
2. $\text{R}$ transforms chemically into another electroactive intermediate $\text{Y}$.
3. $\text{Y}$ is reduced at the electrode surface to form product $\text{Z}$.

In [8]:
spec_O = Species(name="O", D=1.0e-5, c_bulk=1.0e-6)
spec_R = Species(name="R", D=1.0e-5, c_bulk=0.0)
spec_Y = Species(name="Y", D=1.0e-5, c_bulk=0.0)
spec_Z = Species(name="Z", D=1.0e-5, c_bulk=0.0)

rxn_E1 = ElectrochemicalReaction(reactants=[spec_O], products=[spec_R], n_electrons=1, E0=0.0)
rxn_C = ChemicalReaction(reactants=[spec_R], products=[spec_Y])
rxn_E2 = ElectrochemicalReaction(reactants=[spec_Y], products=[spec_Z], n_electrons=1, E0=-0.20)

mech_ECE = Mechanism([rxn_E1, rxn_C, rxn_E2])
print(mech_ECE)

Mechanism:
  Species: Species O (D=1.00e-05 cm²/s, c_bulk=1.00e-06 mol/cm³, z=0), Species R (D=1.00e-05 cm²/s, c_bulk=0.00e+00 mol/cm³, z=0), Species Y (D=1.00e-05 cm²/s, c_bulk=0.00e+00 mol/cm³, z=0), Species Z (D=1.00e-05 cm²/s, c_bulk=0.00e+00 mol/cm³, z=0)
  Electrochemical Reactions:
  O + e⁻ ⇌ R (E0 = 0.000 V)
  Y + e⁻ ⇌ Z (E0 = -0.200 V)
  Chemical Reactions:
  R ⇌ Y


## Step 7: The DISP Mechanism (Disproportionation Pathways)

In many practical electrochemical systems, the ECE mechanism competes directly with a **disproportionation (DISP)** pathway.

Rather than intermediate $\text{Y}$ being reduced heterogeneously at the electrode, it can undergo **homogeneous electron transfer** in the diffusion layer by reacting directly with $\text{R}$.

### Variant A: ECE vs. DISP Competition
$$\text{O} + \text{e}^- \rightleftharpoons \text{R} \quad (E_0 = 0.00\text{ V})$$
$$\text{R} \rightleftharpoons \text{Y}$$
$$\text{R} + \text{Y} \rightleftharpoons \text{O} + \text{Z} \quad (\text{Homogeneous electron exchange})$$

Here, the reaction $\text{R} + \text{Y} \rightleftharpoons \text{O} + \text{Z}$ has multiple reactants and products. Soft Potato supports multiple species in chemical reactions directly.

In [9]:
spec_O = Species(name="O", D=1.0e-5, c_bulk=1.0e-6)
spec_R = Species(name="R", D=1.0e-5, c_bulk=0.0)
spec_Y = Species(name="Y", D=1.0e-5, c_bulk=0.0)
spec_Z = Species(name="Z", D=1.0e-5, c_bulk=0.0)

rxn_E = ElectrochemicalReaction(reactants=[spec_O], products=[spec_R], n_electrons=1, E0=0.0)
rxn_C = ChemicalReaction(reactants=[spec_R], products=[spec_Y])

# Homogeneous disproportionation / electron exchange: R + Y ⇌ O + Z
rxn_disp = ChemicalReaction(
    reactants=[spec_R, spec_Y],
    products=[spec_O, spec_Z],
    stoich_reactants=[1, 1],
    stoich_products=[1, 1],
)

mech_DISP = Mechanism([rxn_E, rxn_C, rxn_disp])
print(mech_DISP)
print("\nHomogeneous Stoichiometry Matrix (ν_i,j):")
print("Species:", mech_DISP.species_names)
print(mech_DISP.homogeneous_stoichiometry_matrix)

Mechanism:
  Species: Species O (D=1.00e-05 cm²/s, c_bulk=1.00e-06 mol/cm³, z=0), Species R (D=1.00e-05 cm²/s, c_bulk=0.00e+00 mol/cm³, z=0), Species Y (D=1.00e-05 cm²/s, c_bulk=0.00e+00 mol/cm³, z=0), Species Z (D=1.00e-05 cm²/s, c_bulk=0.00e+00 mol/cm³, z=0)
  Electrochemical Reactions:
  O + e⁻ ⇌ R (E0 = 0.000 V)
  Chemical Reactions:
  R ⇌ Y
  R + Y ⇌ O + Z

Homogeneous Stoichiometry Matrix (ν_i,j):
Species: ['O', 'R', 'Y', 'Z']
[[ 0.  1.]
 [-1. -1.]
 [ 1. -1.]
 [ 0.  1.]]


### Variant B: Direct Radical Disproportionation

$$\text{O} + \text{e}^- \rightleftharpoons \text{R} \quad (E_0 = 0.00\text{ V})$$
$$2\text{R} \rightleftharpoons \text{O} + \text{Z}$$

In this pathway, two molecules of intermediate $\text{R}$ (such as electrogenerated radical anions) disproportionate bimolecularly to regenerate $\text{O}$ and yield $\text{Z}$.

We configure this stoichiometry via `stoich_reactants=[2]`.

In [10]:
spec_O = Species(name="O", D=1.0e-5, c_bulk=1.0e-6)
spec_R = Species(name="R", D=1.0e-5, c_bulk=0.0)
spec_Z = Species(name="Z", D=1.0e-5, c_bulk=0.0)

rxn_E = ElectrochemicalReaction(reactants=[spec_O], products=[spec_R], n_electrons=1, E0=0.0)

# 2R ⇌ O + Z
rxn_disp_direct = ChemicalReaction(
    reactants=[spec_R],
    products=[spec_O, spec_Z],
    stoich_reactants=[2],
    stoich_products=[1, 1],
)

mech_DISP_direct = Mechanism([rxn_E, rxn_disp_direct])
print(mech_DISP_direct)
print("\nHomogeneous Stoichiometry Matrix (ν_i,j):")
print("Species:", mech_DISP_direct.species_names)
print(mech_DISP_direct.homogeneous_stoichiometry_matrix)

Mechanism:
  Species: Species O (D=1.00e-05 cm²/s, c_bulk=1.00e-06 mol/cm³, z=0), Species R (D=1.00e-05 cm²/s, c_bulk=0.00e+00 mol/cm³, z=0), Species Z (D=1.00e-05 cm²/s, c_bulk=0.00e+00 mol/cm³, z=0)
  Electrochemical Reactions:
  O + e⁻ ⇌ R (E0 = 0.000 V)
  Chemical Reactions:
  2R ⇌ O + Z

Homogeneous Stoichiometry Matrix (ν_i,j):
Species: ['O', 'R', 'Z']
[[ 1.]
 [-2.]
 [ 1.]]


## Step 8: Advanced Features & Working with Solvers

The `Mechanism` class provides essential utilities for interacting with numerical finite-difference solvers:

1. **Dictionary-like access**: Access species by name (`mech["O"]` or `mech.get_species("O")`).
2. **Reaction indexing**: Access reactions by index (`mech[0]`).
3. **Membership checks**: `'O' in mech` or `spec_O in mech`.
4. **Profile initialization**: Allocate 1D NumPy arrays across spatial nodes using `initialize_profiles(n_nodes)`.
5. **Resetting profiles**: Restore bulk concentrations between simulation runs using `reset()`.

In [11]:
# Accessing species and reactions
print("Access species by name:", mech_DISP["O"])
print("Access reaction by index:", mech_DISP[0])
print("Is 'Z' in mechanism?", "Z" in mech_DISP)

# Memory allocation for spatial discretization (e.g., 100 spatial grid nodes)
mech_DISP.initialize_profiles(n_nodes=100)
print(f"Species O profile shape: {mech_DISP['O'].c_profile.shape}")
print(f"Surface concentration O: {mech_DISP['O'].c_profile[0]} mol/cm³")
print(f"Bulk concentration O:    {mech_DISP['O'].c_profile[-1]} mol/cm³")

# Modifying surface concentration (simulating a reaction step)
mech_DISP["O"].c_profile[0] = 0.0

# Reset back to initial bulk conditions
mech_DISP.reset()
print(f"Surface concentration after reset: {mech_DISP['O'].c_profile[0]} mol/cm³")

Access species by name: Species O (D=1.00e-05 cm²/s, c_bulk=1.00e-06 mol/cm³, z=0)
Access reaction by index: O + e⁻ ⇌ R (E0 = 0.000 V)
Is 'Z' in mechanism? True
Species O profile shape: (100,)
Surface concentration O: 1e-06 mol/cm³
Bulk concentration O:    1e-06 mol/cm³
Surface concentration after reset: 1e-06 mol/cm³


## Summary & Next Steps

You now have a solid understanding of how to define any reaction mechanism in Soft Potato 3.0!

### Key Takeaways:
- **CGS Units**: Always specify diffusion coefficients in $\text{cm}^2/\text{s}$ and bulk concentrations in $\text{mol}/\text{cm}^3$ ($1\text{ mM} = 10^{-6}\text{ mol}/\text{cm}^3$).
- **Reactions**: Use `ElectrochemicalReaction` for interfacial electron transfers ($O + n e^- \rightleftharpoons R$) and `ChemicalReaction` for bulk transformations.
- **Mechanism**: Group reactions into a `Mechanism`, which automatically validates species consistency, calculates the stoichiometry matrix, and coordinates memory management across species profiles.

### What's Next:
To simulate cyclic voltammograms or potential step transients for these mechanisms, combine your `Mechanism` with an electrode geometry from `softpotato.geometry` and a technique from `softpotato.techniques`:
```python
# Example workflow:
# grid = sp.geometry.UniformGrid(x_max=0.05, nodes=500)
# electrode = sp.geometry.Planar(area=0.0707, grid=grid)
# cv = sp.techniques.CyclicVoltammetry(E_initial=0.5, E_vertex1=-0.5, scan_rate=0.1)
# sim = sp.simulate.Solver(mech_EC, electrode, cv, method="EFD")
# results = sim.run()
```